# Feature Consistency Analysis Across Random Seeds

This notebook analyzes how consistent the learned features are across different random seeds for Sparse Autoencoders trained on MNIST.


In [ ]:
import os
import re
import glob
from pathlib import Path
from collections import defaultdict
from itertools import combinations

import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt
from scipy.optimize import linear_sum_assignment
import seaborn as sns

# Set style
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette("husl")


def repo_root() -> Path:
    """Return the repository root containing downloaded artifact directories."""
    p = Path.cwd().resolve()
    for cand in [p, *p.parents]:
        if (cand / "data_model_weights").is_dir() or (cand / "scripts" / "download_artifact.py").is_file():
            return cand
    raise FileNotFoundError("Could not find repo root. Run this notebook from inside the repository.")


REPO_ROOT = repo_root()
DATA_MODEL_WEIGHTS_DIR = REPO_ROOT / "data_model_weights"
EXTERNAL_DATA_DIR = REPO_ROOT / "external"


# =============================================================================
# Model class definitions (needed to load saved models)
# =============================================================================

@torch.no_grad()
def set_decoder_norm_to_unit_norm(W_dec, input_dim, hidden_dim):
    D, F = W_dec.shape
    assert D == input_dim, f"Expected input_dim={input_dim}, got {D}"
    assert F == hidden_dim, f"Expected hidden_dim={hidden_dim}, got {F}"
    eps = torch.finfo(W_dec.dtype).eps
    norm = torch.norm(W_dec.data, dim=0, keepdim=True)
    W_dec.data /= norm + eps
    return W_dec.data


class SAE_l1w_encoder_decoder(nn.Module):
    """SAE with L1 weight regularization (needed for unpickling saved models)."""
    def __init__(self, input_dim, hidden_dim, l1=1e-3, l1_w=1e-3, seed=0):
        super().__init__()
        self.input_dim = int(input_dim)
        self.hidden_dim = int(hidden_dim)
        self.l1 = float(l1)
        self.l1_w = float(l1_w)
        self.seed = int(seed)
        self.decoder = nn.Linear(self.hidden_dim, self.input_dim, bias=False)
        self.encoder = nn.Linear(self.input_dim, self.hidden_dim, bias=True)
        self.b_dec = nn.Parameter(torch.zeros(self.input_dim))

    def encode(self, x):
        return F.relu(self.encoder(x - self.b_dec))

    def decode(self, z):
        return self.decoder(z) + self.b_dec

    def forward(self, x):
        z = self.encode(x)
        x_hat = self.decode(z)
        return x_hat, z


class SAE_l2w_encoder_decoder(nn.Module):
    """SAE with L2 weight regularization (needed for unpickling saved models)."""
    def __init__(self, input_dim, hidden_dim, l1=1e-3, l2_w=1e-3, seed=0):
        super().__init__()
        self.input_dim = int(input_dim)
        self.hidden_dim = int(hidden_dim)
        self.l1 = float(l1)
        self.l2_w = float(l2_w)
        self.seed = int(seed)
        self.decoder = nn.Linear(self.hidden_dim, self.input_dim, bias=False)
        self.encoder = nn.Linear(self.input_dim, self.hidden_dim, bias=True)
        self.b_dec = nn.Parameter(torch.zeros(self.input_dim))

    def encode(self, x):
        return F.relu(self.encoder(x - self.b_dec))

    def decode(self, z):
        return self.decoder(z) + self.b_dec

    def forward(self, x):
        z = self.encode(x)
        x_hat = self.decode(z)
        return x_hat, z


## 1. Load Models and Organize by Configuration


In [ ]:
def parse_model_name(filename):
    """
    Parse model filename to extract hyperparameters.
    Example: sae_l1w_l1_0.1_l2w_0.001_100ep_tied_init_seed_2.pt
    """
    basename = os.path.basename(filename)
    
    # Extract seed
    seed_match = re.search(r'seed_(\d+)', basename)
    seed = int(seed_match.group(1)) if seed_match else None
    
    # Extract epochs
    ep_match = re.search(r'(\d+)ep', basename)
    epochs = int(ep_match.group(1)) if ep_match else None
    
    # Extract l1 value
    l1_match = re.search(r'l1_([0-9.]+)_', basename)
    l1 = float(l1_match.group(1)) if l1_match else None
    
    # Extract l2w or l1w value
    l2w_match = re.search(r'l2w_([0-9.]+)', basename)
    l1w_match = re.search(r'l1w_([0-9.]+)(?:_|$)', basename)
    
    if l2w_match:
        weight_reg_type = 'l2w'
        weight_reg_val = float(l2w_match.group(1))
    elif l1w_match:
        weight_reg_type = 'l1w'
        weight_reg_val = float(l1w_match.group(1))
    else:
        weight_reg_type = None
        weight_reg_val = None
    
    # Is tied init?
    tied_init = 'tied_init' in basename
    
    return {
        'seed': seed,
        'epochs': epochs,
        'l1': l1,
        'weight_reg_type': weight_reg_type,
        'weight_reg_val': weight_reg_val,
        'tied_init': tied_init,
        'filename': filename
    }


def group_models_by_config(model_dir):
    """
    Group models by their configuration (excluding seed).
    Returns a dict: config_key -> list of (seed, model_path)
    """
    model_files = glob.glob(os.path.join(model_dir, '**', '*.pt'), recursive=True)
    
    grouped = defaultdict(list)
    
    for f in model_files:
        params = parse_model_name(f)
        # Create config key (everything except seed)
        config_key = (
            params['l1'],
            params['weight_reg_type'],
            params['weight_reg_val'],
            params['epochs'],
            params['tied_init']
        )
        grouped[config_key].append((params['seed'], f))
    
    # Sort by seed within each group
    for key in grouped:
        grouped[key].sort(key=lambda x: x[0])
    
    return dict(grouped)


# Load and group models
model_dirs = [
    DATA_MODEL_WEIGHTS_DIR / "trained_autoencoder_MNISTs" / "trained_autoencoder_MNISTs_squared" / "trained_autoencoder_MNISTs_squared",
    DATA_MODEL_WEIGHTS_DIR / "trained_autoencoder_MNISTs_not_constrain" / "trained_autoencoder_MNISTs_not_constrain_squared" / "trained_autoencoder_MNISTs_not_constrain_squared",
    DATA_MODEL_WEIGHTS_DIR / "trained_autoencoder_MNISTs" / "trained_autoencoder_MNISTs_squared" / "trained_autoencoder_MNISTs",
    DATA_MODEL_WEIGHTS_DIR / "trained_autoencoder_MNISTs_not_constrain" / "trained_autoencoder_MNISTs_not_constrain_squared" / "trained_autoencoder_MNISTs_not_constrain",
]

all_grouped = {}
for model_dir in model_dirs:
    model_dir = str(model_dir)
    if os.path.exists(model_dir):
        grouped = group_models_by_config(model_dir)
        all_grouped.update(grouped)

# Display discovered configurations
print("Discovered configurations (with multiple seeds):")
print("=" * 80)
for config, seeds_files in sorted(all_grouped.items()):
    if len(seeds_files) >= 2:  # Only show configs with multiple seeds
        l1, wreg_type, wreg_val, epochs, tied = config
        seeds = [s for s, _ in seeds_files]
        print(f"l1={l1}, {wreg_type}={wreg_val}, epochs={epochs}, tied_init={tied}")
        print(f"  Seeds: {seeds}")
        print()


## 2. Feature Consistency Metrics

Following Braun et al. (2024), we use **Mean Max Cosine Similarity** as the primary metric:
- For each latent in SAE1, find the maximum cosine similarity with any latent in SAE2
- Average these maxima to get the overall similarity score

We also implement the more conservative **Shared Features** criterion from the paper:
- Compute Hungarian matching on both **encoder** and **decoder** weights
- A feature is "shared" only if both matchings agree AND similarity > 0.7 in both

Key metrics:
1. **Mean Max Cosine Similarity**: Primary metric - does each feature have a good match somewhere?
2. **Fraction Paired (>0.7)**: What fraction of features have max similarity > 0.7?
3. **Shared Features**: Fraction where encoder AND decoder matchings agree with sim > 0.7


In [ ]:
def get_decoder_features(model):
    """Extract decoder features (columns) from a model. Returns shape [hidden_dim, input_dim]."""
    # decoder.weight has shape [input_dim, hidden_dim], so transpose to get features as rows
    return model.decoder.weight.data.T.cpu().numpy()


def get_encoder_features(model):
    """Extract encoder features (rows) from a model. Returns shape [hidden_dim, input_dim]."""
    # encoder.weight has shape [hidden_dim, input_dim]
    return model.encoder.weight.data.cpu().numpy()


def normalize_features(features):
    """Normalize features to unit norm (per row)."""
    norms = np.linalg.norm(features, axis=1, keepdims=True)
    return features / (norms + 1e-8)


def cosine_similarity_matrix(features_a, features_b):
    """
    Compute pairwise cosine similarity between features.
    features_a: [N, D] - N features of dimension D
    features_b: [M, D] - M features of dimension D
    Returns: [N, M] similarity matrix
    """
    a_norm = normalize_features(features_a)
    b_norm = normalize_features(features_b)
    return a_norm @ b_norm.T


def hungarian_matching(sim_matrix):
    """
    Find optimal 1-to-1 matching using Hungarian algorithm.
    Maximizes sum of similarities (not absolute similarities).
    Returns: row_ind, col_ind, matched_sims
    """
    cost_matrix = -sim_matrix  # minimize negative = maximize positive
    row_ind, col_ind = linear_sum_assignment(cost_matrix)
    matched_sims = sim_matrix[row_ind, col_ind]
    return row_ind, col_ind, matched_sims


def compute_feature_consistency_metrics(model_a, model_b, threshold=0.7):
    """
    Compute feature consistency metrics following Braun et al. methodology.
    
    Primary metric: Mean Max Cosine Similarity
    - For each feature in A, find max similarity with any feature in B
    - Average these maxima
    
    Also computes "shared features" using both encoder and decoder matchings.
    """
    # Get both encoder and decoder features
    dec_a = get_decoder_features(model_a)
    dec_b = get_decoder_features(model_b)
    enc_a = get_encoder_features(model_a)
    enc_b = get_encoder_features(model_b)
    
    # Compute similarity matrices (using absolute value to handle sign flips)
    dec_sim = np.abs(cosine_similarity_matrix(dec_a, dec_b))
    enc_sim = np.abs(cosine_similarity_matrix(enc_a, enc_b))
    
    # ==========================================================================
    # PRIMARY METRIC: Mean Max Cosine Similarity
    # ==========================================================================
    # For each feature in A, find max similarity with any feature in B
    max_sim_dec_a = np.max(dec_sim, axis=1)  # [N,] - max sim for each feature in A
    max_sim_dec_b = np.max(dec_sim, axis=0)  # [M,] - max sim for each feature in B
    max_sim_enc_a = np.max(enc_sim, axis=1)
    max_sim_enc_b = np.max(enc_sim, axis=0)
    
    # Mean max cosine similarity (primary metric)
    mean_max_cos_dec = (np.mean(max_sim_dec_a) + np.mean(max_sim_dec_b)) / 2
    mean_max_cos_enc = (np.mean(max_sim_enc_a) + np.mean(max_sim_enc_b)) / 2
    
    # Fraction of features with max_sim > threshold
    frac_paired_dec = np.mean(max_sim_dec_a > threshold)
    frac_paired_enc = np.mean(max_sim_enc_a > threshold)
    
    # ==========================================================================
    # SHARED FEATURES: Hungarian matching on both encoder and decoder
    # ==========================================================================
    # Hungarian matching on decoder
    dec_row, dec_col, dec_matched_sims = hungarian_matching(dec_sim)
    
    # Hungarian matching on encoder
    enc_row, enc_col, enc_matched_sims = hungarian_matching(enc_sim)
    
    # Find shared features: where both matchings agree AND both have sim > threshold
    n_features = len(dec_row)
    shared_mask = np.zeros(n_features, dtype=bool)
    
    for i in range(n_features):
        # Check if decoder and encoder matchings agree
        dec_partner = dec_col[i]
        enc_partner = enc_col[i]
        
        if dec_partner == enc_partner:
            # Check if both similarities are above threshold
            if dec_matched_sims[i] > threshold and enc_matched_sims[i] > threshold:
                shared_mask[i] = True
    
    frac_shared = np.mean(shared_mask)
    
    metrics = {
        # Primary metrics (Mean Max Cosine Similarity)
        'mean_max_cos_dec': mean_max_cos_dec,
        'mean_max_cos_enc': mean_max_cos_enc,
        'mean_max_cos_avg': (mean_max_cos_dec + mean_max_cos_enc) / 2,
        
        # Fraction paired (max sim > threshold)
        'frac_paired_dec': frac_paired_dec,
        'frac_paired_enc': frac_paired_enc,
        
        # Shared features (encoder + decoder agree)
        'frac_shared': frac_shared,
        
        # Distribution of max similarities
        'max_sims_dec': max_sim_dec_a,
        'max_sims_enc': max_sim_enc_a,
        
        # Hungarian matching results (for visualization)
        'hungarian_dec_sims': dec_matched_sims,
        'hungarian_enc_sims': enc_matched_sims,
        'dec_matching': (dec_row, dec_col),
        'enc_matching': (enc_row, enc_col),
        'shared_mask': shared_mask,
        
        # Raw similarity matrices
        'dec_sim_matrix': dec_sim,
        'enc_sim_matrix': enc_sim,
    }
    
    return metrics


def analyze_seed_consistency(model_paths, verbose=True):
    """
    Analyze feature consistency across models trained with different seeds.
    model_paths: list of (seed, path) tuples
    """
    # Load all models
    models = {}
    for seed, path in model_paths:
        if verbose:
            print(f"Loading seed {seed}: {os.path.basename(path)}")
        models[seed] = torch.load(path, map_location='cpu', weights_only=False)
    
    seeds = list(models.keys())
    
    # Analyze all pairs
    results = {}
    for seed_a, seed_b in combinations(seeds, 2):
        metrics = compute_feature_consistency_metrics(models[seed_a], models[seed_b])
        results[(seed_a, seed_b)] = metrics
        
        if verbose:
            print(f"\nSeed {seed_a} vs Seed {seed_b}:")
            print(f"  Mean Max Cos (decoder): {metrics['mean_max_cos_dec']:.4f}")
            print(f"  Mean Max Cos (encoder): {metrics['mean_max_cos_enc']:.4f}")
            print(f"  Frac paired >0.7 (dec): {metrics['frac_paired_dec']:.2%}")
            print(f"  Frac shared (both):     {metrics['frac_shared']:.2%}")
    
    return results, models


## 3. Visualization Functions


In [ ]:
def plot_max_similarity_distribution(results, config_name=""):
    """Plot distribution of MAX similarities (primary metric) across seed pairs."""
    n_pairs = len(results)
    fig, axes = plt.subplots(2, n_pairs, figsize=(5*n_pairs, 8))
    if n_pairs == 1:
        axes = axes.reshape(2, 1)
    
    for col, ((seed_a, seed_b), metrics) in enumerate(results.items()):
        # Decoder max similarities
        ax = axes[0, col]
        ax.hist(metrics['max_sims_dec'], bins=50, edgecolor='black', alpha=0.7, color='steelblue')
        ax.axvline(metrics['mean_max_cos_dec'], color='red', linestyle='--', 
                   label=f'Mean: {metrics["mean_max_cos_dec"]:.3f}')
        ax.axvline(0.7, color='green', linestyle=':', linewidth=2, label='Threshold 0.7')
        ax.set_xlabel('Max Cosine Similarity')
        ax.set_ylabel('Count')
        ax.set_title(f'Decoder - Seed {seed_a} vs {seed_b}')
        ax.legend(fontsize=8)
        ax.set_xlim(0, 1)
        
        # Encoder max similarities
        ax = axes[1, col]
        ax.hist(metrics['max_sims_enc'], bins=50, edgecolor='black', alpha=0.7, color='coral')
        ax.axvline(metrics['mean_max_cos_enc'], color='red', linestyle='--', 
                   label=f'Mean: {metrics["mean_max_cos_enc"]:.3f}')
        ax.axvline(0.7, color='green', linestyle=':', linewidth=2, label='Threshold 0.7')
        ax.set_xlabel('Max Cosine Similarity')
        ax.set_ylabel('Count')
        ax.set_title(f'Encoder - Seed {seed_a} vs {seed_b}')
        ax.legend(fontsize=8)
        ax.set_xlim(0, 1)
    
    plt.suptitle(f'Max Cosine Similarity Distribution\n{config_name}', y=1.02)
    plt.tight_layout()
    return fig


def plot_similarity_heatmap(sim_matrix, seed_a, seed_b, title_suffix="", max_features=100):
    """Plot heatmap of similarity matrix (subsampled if too large)."""
    n, m = sim_matrix.shape
    
    # Subsample if too large
    if n > max_features or m > max_features:
        step_n = max(1, n // max_features)
        step_m = max(1, m // max_features)
        sim_sub = sim_matrix[::step_n, ::step_m]
    else:
        sim_sub = sim_matrix
    
    fig, ax = plt.subplots(figsize=(10, 8))
    im = ax.imshow(sim_sub, cmap='viridis', aspect='auto', vmin=0, vmax=1)
    plt.colorbar(im, ax=ax, label='|Cosine Similarity|')
    ax.set_xlabel(f'Features (Seed {seed_b})')
    ax.set_ylabel(f'Features (Seed {seed_a})')
    ax.set_title(f'Feature Similarity Matrix {title_suffix}\n(Seed {seed_a} vs {seed_b})')
    return fig


def plot_matched_features(models, metrics, seed_a, seed_b, n_examples=10, show_shared_only=False):
    """Visualize pairs of matched features side-by-side."""
    features_a = get_decoder_features(models[seed_a])
    features_b = get_decoder_features(models[seed_b])
    
    row_ind, col_ind = metrics['dec_matching']
    max_sims = metrics['max_sims_dec']
    shared_mask = metrics['shared_mask']
    
    if show_shared_only:
        # Only show shared features
        indices = np.where(shared_mask)[0]
        title_suffix = "(Shared Features Only)"
    else:
        # Sort by max similarity to show best matches first
        indices = np.argsort(max_sims)[::-1]
        title_suffix = "(Best Matches)"
    
    n_show = min(n_examples, len(indices))
    fig, axes = plt.subplots(2, n_show, figsize=(2*n_show, 5))
    
    for i in range(n_show):
        idx = indices[i]
        idx_a = row_ind[idx]
        idx_b = col_ind[idx]
        sim = max_sims[idx]
        is_shared = shared_mask[idx]
        
        feat_a = features_a[idx_a].reshape(28, 28)
        feat_b = features_b[idx_b].reshape(28, 28)
        
        # Plot feature from seed A
        ax = axes[0, i]
        vmax = max(np.abs(feat_a).max(), 1e-6)
        ax.imshow(feat_a, cmap='bwr', vmin=-vmax, vmax=vmax)
        shared_str = "✓" if is_shared else ""
        ax.set_title(f'Seed {seed_a} #{idx_a}\nsim={sim:.2f} {shared_str}', fontsize=8)
        ax.axis('off')
        
        # Plot matched feature from seed B
        ax = axes[1, i]
        vmax = max(np.abs(feat_b).max(), 1e-6)
        ax.imshow(feat_b, cmap='bwr', vmin=-vmax, vmax=vmax)
        ax.set_title(f'Seed {seed_b} #{idx_b}', fontsize=8)
        ax.axis('off')
    
    plt.suptitle(f'Matched Features {title_suffix}\nSeed {seed_a} vs {seed_b}', y=1.02)
    plt.tight_layout()
    return fig


def plot_feature_grid(model, n_features=50, title=""):
    """Plot grid of decoder features as images."""
    features = get_decoder_features(model)
    
    n_cols = 10
    n_rows = (n_features + n_cols - 1) // n_cols
    
    fig, axes = plt.subplots(n_rows, n_cols, figsize=(n_cols * 1.5, n_rows * 1.5))
    
    for i in range(n_features):
        row = i // n_cols
        col = i % n_cols
        ax = axes[row, col] if n_rows > 1 else axes[col]
        
        feat = features[i].reshape(28, 28)
        vmax = max(np.abs(feat).max(), 1e-6)
        ax.imshow(feat, cmap='bwr', vmin=-vmax, vmax=vmax)
        ax.axis('off')
    
    # Hide empty subplots
    for i in range(n_features, n_rows * n_cols):
        row = i // n_cols
        col = i % n_cols
        ax = axes[row, col] if n_rows > 1 else axes[col]
        ax.axis('off')
    
    plt.suptitle(title, y=1.01)
    plt.tight_layout()
    return fig


## 4. Run Analysis Across All Configurations


In [ ]:
# Run analysis on all configurations with multiple seeds
all_analysis_results = {}

for config, seed_files in sorted(all_grouped.items()):
    if len(seed_files) < 2:
        continue
    
    l1, wreg_type, wreg_val, epochs, tied = config
    config_name = f"l1={l1}, {wreg_type}={wreg_val}, epochs={epochs}, tied={tied}"
    
    print("=" * 80)
    print(f"Analyzing: {config_name}")
    print("=" * 80)
    
    results, models = analyze_seed_consistency(seed_files, verbose=True)
    all_analysis_results[config] = {'results': results, 'models': models, 'name': config_name}
    print()


## 5. Summary Table


In [ ]:
# Create summary table with new metrics
summary_data = []

for config, data in all_analysis_results.items():
    l1, wreg_type, wreg_val, epochs, tied = config
    
    # Average metrics across all seed pairs
    all_mean_max_cos = []
    all_frac_paired = []
    all_frac_shared = []
    
    for (seed_a, seed_b), metrics in data['results'].items():
        all_mean_max_cos.append(metrics['mean_max_cos_avg'])
        all_frac_paired.append(metrics['frac_paired_dec'])
        all_frac_shared.append(metrics['frac_shared'])
    
    summary_data.append({
        'L1': l1,
        'Weight Reg': f"{wreg_type}={wreg_val}",
        'Epochs': epochs,
        'Tied Init': tied,
        'Mean Max Cos': np.mean(all_mean_max_cos),
        'Std': np.std(all_mean_max_cos),
        'Frac Paired (>0.7)': np.mean(all_frac_paired),
        'Frac Shared': np.mean(all_frac_shared),
    })

# Display as formatted table
import pandas as pd
df_summary = pd.DataFrame(summary_data)
df_summary = df_summary.sort_values('Mean Max Cos', ascending=False)
print("\nSummary of Feature Consistency Across Seeds:")
print("=" * 100)
print("Mean Max Cos = Mean Max Cosine Similarity (primary metric)")
print("Frac Paired = fraction of features with max_sim > 0.7")
print("Frac Shared = fraction where encoder & decoder matchings agree with sim > 0.7")
print()
display(df_summary.style.format({
    'Mean Max Cos': '{:.4f}',
    'Std': '{:.4f}',
    'Frac Paired (>0.7)': '{:.2%}',
    'Frac Shared': '{:.2%}',
}).background_gradient(subset=['Mean Max Cos', 'Frac Paired (>0.7)'], cmap='RdYlGn'))


## 7. Visualization: Matched Feature Pairs


In [ ]:
# Visualize unmatched (random) features for all l1/l2 groups with epochs=100 (both tied and untied groups)
# Then also visualize matched feature pairs.

import random


def plot_matched_features_no_text(models, metrics, seed_a, seed_b, n_examples=10, show_shared_only=False):
    """Visualize pairs of matched features side-by-side (row 1: seed 0, row 2: seed 2)."""
    # Force seed_a = 0, seed_b = 2
    seed_a = 0
    seed_b = 2

    features_a = get_decoder_features(models[seed_a])
    features_b = get_decoder_features(models[seed_b])
    
    row_ind, col_ind = metrics['dec_matching']
    max_sims = metrics['max_sims_dec']
    shared_mask = metrics['shared_mask']
    
    if show_shared_only:
        # Only show shared features
        indices = np.where(shared_mask)[0]
        title_suffix = "(Shared Features Only)"
    else:
        # Sort by max similarity to show best matches first
        indices = np.argsort(max_sims)[::-1]
        title_suffix = "(Best Matches)"
    
    n_show = min(n_examples, len(indices))
    fig, axes = plt.subplots(2, n_show, figsize=(2*n_show, 5))
    
    for i in range(n_show):
        idx = indices[i]
        idx_a = row_ind[idx]
        idx_b = col_ind[idx]
        sim = max_sims[idx]
        is_shared = shared_mask[idx]
        
        feat_a = features_a[idx_a].reshape(28, 28)
        feat_b = features_b[idx_b].reshape(28, 28)
        
        # Plot feature from seed 0
        ax = axes[0, i]
        vmax = max(np.abs(feat_a).max(), 1e-6)
        ax.imshow(feat_a, cmap='bwr', vmin=-vmax, vmax=vmax)
        ax.axis('off')
        
        # Plot matched feature from seed 2
        ax = axes[1, i]
        vmax = max(np.abs(feat_b).max(), 1e-6)
        ax.imshow(feat_b, cmap='bwr', vmin=-vmax, vmax=vmax)
        ax.axis('off')
    
    plt.suptitle(f'Matched Features {title_suffix}\nSeed 0 vs Seed 2', y=1.02)
    plt.tight_layout()
    return fig


def plot_random_features(models, metrics, seed_a, seed_b, n_examples=10, feature_type='decoder'):
    """
    Plot random (unmatched) feature pairs across two models.
    Useful as a comparison to see what random pairs look like vs the matched ones.

    In this rewrite, row 1 is always seed 0, row 2 is always seed 2.
    
    Args:
        models: dict of seed -> model
        metrics: metrics dict (not used here but kept for API consistency)
        seed_a, seed_b: (ignored, always taken as 0 and 2)
        n_examples: number of random pairs to show
        feature_type: 'decoder' or 'encoder' - which features to visualize
    """
    seed_a = 0
    seed_b = 2
    # Select feature extraction function based on feature_type
    if feature_type == 'encoder':
        features_a = get_encoder_features(models[seed_a])
        features_b = get_encoder_features(models[seed_b])
    else:  # default to decoder
        features_a = get_decoder_features(models[seed_a])
        features_b = get_decoder_features(models[seed_b])
    
    num_features = min(features_a.shape[0], features_b.shape[0])
    if n_examples > num_features:
        n_examples = num_features

    # Pick random pairs (different indices in each model)
    idxs_a = random.sample(range(num_features), n_examples)
    idxs_b = random.sample(range(num_features), n_examples)
    
    # Compute cosine similarities for the random pairs
    random_sims = []
    for idx_a, idx_b in zip(idxs_a, idxs_b):
        feat_a = features_a[idx_a]
        feat_b = features_b[idx_b]
        # Compute cosine similarity
        norm_a = np.linalg.norm(feat_a)
        norm_b = np.linalg.norm(feat_b)
        if norm_a > 1e-8 and norm_b > 1e-8:
            sim = np.abs(np.dot(feat_a, feat_b) / (norm_a * norm_b))
        else:
            sim = 0.0
        random_sims.append(sim)

    # Create figure
    fig, axes = plt.subplots(2, n_examples, figsize=(2*n_examples, 5))
    
    for i in range(n_examples):
        idx_a = idxs_a[i]
        idx_b = idxs_b[i]
        #sim = random_sims[i]
        
        feat_a = features_a[idx_a].reshape(28, 28)
        feat_b = features_b[idx_b].reshape(28, 28)
        
        # Plot feature from seed 0
        ax = axes[0, i]
        vmax = max(np.abs(feat_a).max(), 1e-6)
        ax.imshow(feat_a, cmap='bwr', vmin=-vmax, vmax=vmax)
        ax.set_title('')
        ax.axis('off')
        
        # Plot random feature from seed 2
        ax = axes[1, i]
        vmax = max(np.abs(feat_b).max(), 1e-6)
        ax.imshow(feat_b, cmap='bwr', vmin=-vmax, vmax=vmax)
        ax.set_title('')
        ax.axis('off')
    
    plt.suptitle(f'RANDOM FEATURE PAIRS (UNMATCHED) - {feature_type.upper()}\nSeed 0 vs Seed 2', fontsize=24, y=1.02)
    plt.tight_layout()
    return fig

def visualize_matched_features_group(tied_value, feature_type='decoder'):
    """
    Visualize matched and random features for a group of configurations.

    Args:
        tied_value: True for tied init models, False for untied
        feature_type: 'decoder' or 'encoder' - which features to visualize
    """
    print(f"\n{'='*40}\n{'TIED' if tied_value else 'UNTIED'} group ({feature_type} features)\n{'='*40}")
    found_any = False
    for config, data in all_analysis_results.items():
        l1, wreg_type, wreg_val, epochs, tied = config
        if epochs != 100 or tied != tied_value:
            continue

        found_any = True
        models = data['models']
        results = data['results']

        # Use always seed_a=0, seed_b=2 if available
        seeds = list(models.keys())
        if 0 in seeds and 2 in seeds:
            seed_a = 0
            seed_b = 2
        else:
            # fallback to the first available pair in results
            (seed_a, seed_b), metrics = list(results.items())[0]

        # Ensure metrics correspond to (0,2) if possible
        if (0,2) in results:
            metrics = results[(0,2)]
        else:
            # fallback to any
            (seed_a, seed_b), metrics = list(results.items())[0]
            seed_a = 0
            seed_b = 2

        print(f"\n{data['name']}")
        print(f"Mean Max Cos: {metrics['mean_max_cos_avg']:.4f}")
        print(f"Frac Shared: {metrics['frac_shared']:.2%}")

        # 1. Show random feature pairs for a benchmark FIRST (rows: seed 0/top, seed 2/bottom)
        fig = plot_random_features(models, metrics, seed_a, seed_b, n_examples=10, feature_type=feature_type)
        plt.show()

        # 2. Show best matched features
        fig = plot_matched_features_no_text(models, metrics, seed_a, seed_b, n_examples=10, show_shared_only=False)
        plt.suptitle('MATCHED FEATURE PAIRS', fontsize=24, y=1.02)
        plt.tight_layout()
        plt.show()

        # 3. Show shared features only (if any)
        if metrics['frac_shared'] > 0:
            fig = plot_matched_features_no_text(models, metrics, seed_a, seed_b, n_examples=10, show_shared_only=True)
            plt.suptitle('SHARED FEATURE PAIRS (MATCHED, SHARED)', fontsize=24, y=1.02)
            plt.tight_layout()
            plt.show()

    if not found_any:
        print("No configurations found for this group.")

# ============================================
# CONFIGURATION: Choose which features to plot
# ============================================
FEATURE_TYPE = 'encoder'  # 'decoder' or 'encoder'

# Show for tied group (l1/l2/other with tied weights)
visualize_matched_features_group(tied_value=True, feature_type=FEATURE_TYPE)
# Show for untied group (l1/l2/other with untied weights)
visualize_matched_features_group(tied_value=False, feature_type=FEATURE_TYPE)


## 8. Comparison: Impact of Weight Regularization on Feature Consistency

We compare three groups of models (all with l1=0.1, epochs=100, seeds=[0,1,2]):

1. **No weight regularization**: l2w=0.0 (baseline)
2. **L2 weight regularization**: l2w=0.001
3. **L1 weight regularization**: l1w=0.001

Each group includes both tied_init=True and tied_init=False configurations.


In [ ]:
# Compare consistency across different regularization settings
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# Prepare data for plotting
configs_100ep = [(c, d) for c, d in all_analysis_results.items() if c[3] == 100]  # epochs=100

if configs_100ep:
    labels = []
    mean_max_cos = []
    frac_paired = []
    frac_shared = []
    
    for config, data in sorted(configs_100ep, key=lambda x: x[0]):
        l1, wreg_type, wreg_val, epochs, tied = config
        label = f"{wreg_type}={wreg_val}"
        if tied:
            label += " (tied)"
        labels.append(label)
        
        # Average across seed pairs
        mean_max_cos.append(np.mean([m['mean_max_cos_avg'] for m in data['results'].values()]))
        frac_paired.append(np.mean([m['frac_paired_dec'] for m in data['results'].values()]))
        frac_shared.append(np.mean([m['frac_shared'] for m in data['results'].values()]))
    
    x = np.arange(len(labels))
    width = 0.6
    
    # Mean Max Cosine Similarity
    bars1 = axes[0].bar(x, mean_max_cos, width, color='steelblue', edgecolor='black')
    axes[0].set_ylabel('Mean Max Cosine Similarity')
    axes[0].set_xlabel('Configuration')
    axes[0].set_title('Primary Metric: Mean Max Cos Sim')
    axes[0].set_xticks(x)
    axes[0].set_xticklabels(labels, rotation=45, ha='right')
    axes[0].set_ylim(0, 1)
    
    # Fraction Paired (>0.7)
    bars2 = axes[1].bar(x, frac_paired, width, color='coral', edgecolor='black')
    axes[1].set_ylabel('Fraction Paired (>0.7)')
    axes[1].set_xlabel('Configuration')
    axes[1].set_title('Features with Max Sim > 0.7')
    axes[1].set_xticks(x)
    axes[1].set_xticklabels(labels, rotation=45, ha='right')
    axes[1].set_ylim(0, 1)
    
    # Fraction Shared (conservative)
    bars3 = axes[2].bar(x, frac_shared, width, color='forestgreen', edgecolor='black')
    axes[2].set_ylabel('Fraction Shared')
    axes[2].set_xlabel('Configuration')
    axes[2].set_title('Shared (Enc+Dec agree, >0.7)')
    axes[2].set_xticks(x)
    axes[2].set_xticklabels(labels, rotation=45, ha='right')
    axes[2].set_ylim(0, 1)
    
    plt.suptitle('Feature Consistency Comparison (100 epochs)', y=1.02)
    plt.tight_layout()
    plt.show()
else:
    print("No 100-epoch configurations found")


## 12. Conclusions

This analysis measures how consistent the learned sparse features are across different random seeds, following methodology from Braun et al. (2024).

**Primary Metric: Mean Max Cosine Similarity**
- For each feature in SAE1, find the maximum cosine similarity with any feature in SAE2
- Average these maxima to get the overall similarity score
- This answers: "Does each feature have a good match somewhere?"

**Secondary Metrics:**
- **Frac Paired (>0.7)**: Fraction of features with max similarity > 0.7
- **Frac Shared**: Conservative metric where encoder AND decoder Hungarian matchings must agree with similarity > 0.7 in both

**Interpretation:**
- High fraction paired (>80%) suggests features are learning true data structure
- Braun et al. found ~42% shared features across independently trained LLM SAEs
- Lower consistency may indicate features are more dependent on initialization


In [ ]:
# Final summary
print("=" * 80)
print("FINAL SUMMARY: Feature Consistency Across Random Seeds")
print("=" * 80)

if all_analysis_results:
    # Find best and worst configurations by Mean Max Cosine Similarity
    best_config = max(all_analysis_results.items(), 
                      key=lambda x: np.mean([m['mean_max_cos_avg'] for m in x[1]['results'].values()]))
    worst_config = min(all_analysis_results.items(),
                       key=lambda x: np.mean([m['mean_max_cos_avg'] for m in x[1]['results'].values()]))
    
    best_mean_max = np.mean([m['mean_max_cos_avg'] for m in best_config[1]['results'].values()])
    best_frac_paired = np.mean([m['frac_paired_dec'] for m in best_config[1]['results'].values()])
    best_frac_shared = np.mean([m['frac_shared'] for m in best_config[1]['results'].values()])
    
    worst_mean_max = np.mean([m['mean_max_cos_avg'] for m in worst_config[1]['results'].values()])
    worst_frac_paired = np.mean([m['frac_paired_dec'] for m in worst_config[1]['results'].values()])
    
    print(f"\n✓ Most consistent features:")
    print(f"  {best_config[1]['name']}")
    print(f"  Mean Max Cos Sim: {best_mean_max:.4f}")
    print(f"  Frac Paired (>0.7): {best_frac_paired:.2%}")
    print(f"  Frac Shared: {best_frac_shared:.2%}")
    
    print(f"\n✗ Least consistent features:")
    print(f"  {worst_config[1]['name']}")
    print(f"  Mean Max Cos Sim: {worst_mean_max:.4f}")
    print(f"  Frac Paired (>0.7): {worst_frac_paired:.2%}")
    
    print("\n" + "=" * 80)
    print("Interpretation (using 0.7 threshold from Braun et al.):")
    print("-" * 80)
    if best_frac_paired > 0.8:
        print("• High consistency: >80% of features have a good match (max sim > 0.7)")
        print("• Features are likely capturing true structure in the data")
    elif best_frac_paired > 0.5:
        print("• Moderate consistency: 50-80% of features have a good match")
        print("• Some features are stable, others are seed-dependent")
    else:
        print("• Low consistency: <50% of features have a good match")
        print("• Features are highly dependent on initialization")
    
    print(f"\n• Shared features (enc+dec agree): {best_frac_shared:.1%} of latents")
    print("  (This is comparable to ~42% found in Braun et al. for LLM SAEs)")
else:
    print("No analysis results available")


## 13. Dead Feature Analysis: Encoder-Decoder Alignment

Dead or poorly trained features might have arbitrary weights that don't contribute to reconstruction. We can identify "alive" features by measuring the cosine similarity between each feature's encoder and decoder weights within the same model.

**Intuition**: For a well-trained feature:
- The encoder weight detects the feature pattern in inputs
- The decoder weight reconstructs that same pattern
- These should be well-aligned (high cosine similarity)

Dead features may have diverged encoder/decoder weights since they're never activated during training.


In [ ]:
def compute_encoder_decoder_alignment(model):
    """
    Compute cosine similarity between encoder and decoder weights for each feature.
    
    For feature i:
    - Encoder weight: encoder.weight[i, :] (how it detects the feature)
    - Decoder weight: decoder.weight[:, i] (how it reconstructs the feature)
    
    Returns array of shape [hidden_dim] with cosine similarities.
    """
    enc = model.encoder.weight.data.cpu().numpy()  # [hidden_dim, input_dim]
    dec = model.decoder.weight.data.cpu().numpy()  # [input_dim, hidden_dim]
    
    # Normalize each feature's weights
    enc_norm = enc / (np.linalg.norm(enc, axis=1, keepdims=True) + 1e-8)
    dec_norm = dec / (np.linalg.norm(dec, axis=0, keepdims=True) + 1e-8)  # normalize columns
    
    # Compute cosine similarity for each feature (dot product of normalized vectors)
    # enc_norm[i, :] dot dec_norm[:, i] = sum(enc_norm[i, :] * dec_norm[:, i])
    alignment = np.sum(enc_norm * dec_norm.T, axis=1)  # [hidden_dim]
    
    return alignment


def get_alive_feature_mask(model, threshold=0.5):
    """
    Get mask of 'alive' features based on encoder-decoder alignment.
    Features with alignment > threshold are considered alive.
    """
    alignment = compute_encoder_decoder_alignment(model)
    return alignment > threshold, alignment


# Compute encoder-decoder alignment for all models
print("Encoder-Decoder Alignment Analysis")
print("=" * 80)

alignment_stats = []

for config, data in all_analysis_results.items():
    l1, wreg_type, wreg_val, epochs, tied = config
    if epochs != 100:
        continue
    
    models = data['models']
    config_name = data['name']
    
    for seed, model in models.items():
        alignment = compute_encoder_decoder_alignment(model)
        
        stats = {
            'config': config_name,
            'seed': seed,
            'mean_alignment': np.mean(alignment),
            'median_alignment': np.median(alignment),
            'frac_above_0.5': np.mean(alignment > 0.5),
            'frac_above_0.7': np.mean(alignment > 0.7),
            'frac_above_0.9': np.mean(alignment > 0.9),
            'alignment': alignment
        }
        alignment_stats.append(stats)
        
print(f"Analyzed {len(alignment_stats)} models")
print()


## 14. Re-computing Cross-Seed Consistency with Alive Features Only

Now we re-run the consistency analysis but only considering features that are "alive" in both models being compared. This should give us a cleaner picture of whether the *meaningful* features are consistent across seeds.


In [ ]:
def compute_filtered_consistency(model_a, model_b, alignment_threshold=0.2, similarity_threshold=0.7):
    """
    Compute feature consistency only for features that are 'alive' in BOTH models.
    Uses the SAME methodology as compute_feature_consistency_metrics (Braun et al.).
    
    A feature is 'alive' if its encoder-decoder alignment > alignment_threshold.
    Default threshold is 0.2 to only filter truly dead/divergent features.
    """
    # Get alignment for both models
    align_a = compute_encoder_decoder_alignment(model_a)
    align_b = compute_encoder_decoder_alignment(model_b)
    
    alive_mask_a = align_a > alignment_threshold
    alive_mask_b = align_b > alignment_threshold
    
    # Get features
    dec_a = get_decoder_features(model_a)
    dec_b = get_decoder_features(model_b)
    enc_a = get_encoder_features(model_a)
    enc_b = get_encoder_features(model_b)
    
    # Filter to alive features only
    dec_a_alive = dec_a[alive_mask_a]
    dec_b_alive = dec_b[alive_mask_b]
    enc_a_alive = enc_a[alive_mask_a]
    enc_b_alive = enc_b[alive_mask_b]
    
    n_alive_a = int(alive_mask_a.sum())
    n_alive_b = int(alive_mask_b.sum())
    n_total = len(align_a)
    
    if n_alive_a == 0 or n_alive_b == 0:
        return {
            'n_alive_a': n_alive_a,
            'n_alive_b': n_alive_b,
            'n_total': n_total,
            'frac_alive_a': 0.0,
            'frac_alive_b': 0.0,
            'mean_max_cos_dec': np.nan,
            'mean_max_cos_enc': np.nan,
            'mean_max_cos_avg': np.nan,
            'frac_paired_dec': np.nan,
            'frac_paired_enc': np.nan,
            'frac_shared': np.nan,
            'max_sims_dec': np.array([]),
            'max_sims_enc': np.array([]),
        }
    
    # Compute similarity matrices for alive features only (using abs for sign flips)
    dec_sim = np.abs(cosine_similarity_matrix(dec_a_alive, dec_b_alive))
    enc_sim = np.abs(cosine_similarity_matrix(enc_a_alive, enc_b_alive))
    
    # ==========================================================================
    # PRIMARY METRIC: Mean Max Cosine Similarity (same as original)
    # ==========================================================================
    max_sim_dec_a = np.max(dec_sim, axis=1)
    max_sim_dec_b = np.max(dec_sim, axis=0)
    max_sim_enc_a = np.max(enc_sim, axis=1)
    max_sim_enc_b = np.max(enc_sim, axis=0)
    
    mean_max_cos_dec = (np.mean(max_sim_dec_a) + np.mean(max_sim_dec_b)) / 2
    mean_max_cos_enc = (np.mean(max_sim_enc_a) + np.mean(max_sim_enc_b)) / 2
    
    # Fraction paired
    frac_paired_dec = np.mean(max_sim_dec_a > similarity_threshold)
    frac_paired_enc = np.mean(max_sim_enc_a > similarity_threshold)
    
    # ==========================================================================
    # SHARED FEATURES: Hungarian matching on both encoder and decoder (same as original)
    # ==========================================================================
    dec_row, dec_col, dec_matched_sims = hungarian_matching(dec_sim)
    enc_row, enc_col, enc_matched_sims = hungarian_matching(enc_sim)
    
    # Find shared features: where both matchings agree AND both have sim > threshold
    n_features_to_match = min(n_alive_a, n_alive_b)
    shared_mask = np.zeros(n_features_to_match, dtype=bool)
    
    for i in range(n_features_to_match):
        dec_partner = dec_col[i]
        enc_partner = enc_col[i]
        
        if dec_partner == enc_partner:
            if dec_matched_sims[i] > similarity_threshold and enc_matched_sims[i] > similarity_threshold:
                shared_mask[i] = True
    
    frac_shared = np.mean(shared_mask) if len(shared_mask) > 0 else 0.0
    
    return {
        'n_alive_a': n_alive_a,
        'n_alive_b': n_alive_b,
        'n_total': n_total,
        'frac_alive_a': n_alive_a / n_total,
        'frac_alive_b': n_alive_b / n_total,
        # Primary metrics
        'mean_max_cos_dec': mean_max_cos_dec,
        'mean_max_cos_enc': mean_max_cos_enc,
        'mean_max_cos_avg': (mean_max_cos_dec + mean_max_cos_enc) / 2,
        # Fraction paired
        'frac_paired_dec': frac_paired_dec,
        'frac_paired_enc': frac_paired_enc,
        # Shared features (same methodology as original)
        'frac_shared': frac_shared,
        # Distributions
        'max_sims_dec': max_sim_dec_a,
        'max_sims_enc': max_sim_enc_a,
        'hungarian_dec_sims': dec_matched_sims,
        'hungarian_enc_sims': enc_matched_sims,
        'shared_mask': shared_mask,
    }


# Re-run analysis with alive feature filtering
ALIGNMENT_THRESHOLD = 0.1  # Only filter truly dead features (very low enc-dec alignment)

print("Cross-Seed Consistency Analysis: ALIVE FEATURES ONLY")
print("=" * 80)
print(f"(Filtering to features with encoder-decoder alignment > {ALIGNMENT_THRESHOLD})")
print()

filtered_results = {}

for config, data in all_analysis_results.items():
    l1, wreg_type, wreg_val, epochs, tied = config
    if epochs != 100:
        continue
    
    models = data['models']
    config_name = data['name']
    seeds = list(models.keys())
    
    config_filtered = {}
    
    for seed_a, seed_b in combinations(seeds, 2):
        metrics = compute_filtered_consistency(models[seed_a], models[seed_b], 
                                                alignment_threshold=ALIGNMENT_THRESHOLD)
        config_filtered[(seed_a, seed_b)] = metrics
        
        print(f"{config_name} - Seed {seed_a} vs {seed_b}:")
        print(f"  Alive features: {metrics['n_alive_a']}/{metrics['n_total']} vs {metrics['n_alive_b']}/{metrics['n_total']}")
        print(f"  Mean Max Cos (alive only): {metrics['mean_max_cos_avg']:.4f}")
        print(f"  Frac Paired >0.7 (alive):  {metrics['frac_paired_dec']:.2%}")
        print(f"  Frac Shared (alive):       {metrics['frac_shared']:.2%}")
        print()
    
    filtered_results[config] = {'results': config_filtered, 'name': config_name}


In [ ]:
# Compare filtered vs unfiltered results
print("\nComparison: All Features vs Alive Features Only")
print("=" * 80)
print(f"(Alive = encoder-decoder alignment > {ALIGNMENT_THRESHOLD})")
print()

comparison_data = []

for config in filtered_results.keys():
    if config not in all_analysis_results:
        continue
    
    l1, wreg_type, wreg_val, epochs, tied = config
    config_name = filtered_results[config]['name']
    
    # Average over seed pairs - ALL features
    all_mean_max = np.mean([m['mean_max_cos_avg'] for m in all_analysis_results[config]['results'].values()])
    all_frac_paired = np.mean([m['frac_paired_dec'] for m in all_analysis_results[config]['results'].values()])
    all_frac_shared = np.mean([m['frac_shared'] for m in all_analysis_results[config]['results'].values()])
    
    # Average over seed pairs - ALIVE features only
    filtered_metrics = [m for m in filtered_results[config]['results'].values() if not np.isnan(m['mean_max_cos_avg'])]
    filtered_mean_max = np.mean([m['mean_max_cos_avg'] for m in filtered_metrics])
    filtered_frac_paired = np.mean([m['frac_paired_dec'] for m in filtered_metrics])
    filtered_frac_shared = np.mean([m['frac_shared'] for m in filtered_metrics])
    
    frac_alive = np.mean([m['frac_alive_a'] for m in filtered_results[config]['results'].values()])
    
    comparison_data.append({
        'Config': config_name.split(',')[1].strip(),
        'Tied': tied,
        'Frac Alive': frac_alive,
        'All: Mean Max Cos': all_mean_max,
        'Alive: Mean Max Cos': filtered_mean_max,
        'Δ Mean Max Cos': filtered_mean_max - all_mean_max,
        'All: Frac Shared': all_frac_shared,
        'Alive: Frac Shared': filtered_frac_shared,
        'Δ Frac Shared': filtered_frac_shared - all_frac_shared,
    })

df_comparison = pd.DataFrame(comparison_data)
print("\nImpact of Dead Feature Filtering (using same methodology as Braun et al.):")
display(df_comparison.style.format({
    'Frac Alive': '{:.1%}',
    'All: Mean Max Cos': '{:.4f}',
    'Alive: Mean Max Cos': '{:.4f}',
    'Δ Mean Max Cos': '{:+.4f}',
    'All: Frac Shared': '{:.1%}',
    'Alive: Frac Shared': '{:.1%}',
    'Δ Frac Shared': '{:+.1%}',
}).background_gradient(subset=['Δ Mean Max Cos', 'Δ Frac Shared'], cmap='RdYlGn'))


In [ ]:
# Compare consistency: ALL FEATURES vs ALIVE FEATURES ONLY (enc-dec alignment > threshold)
fig, axes = plt.subplots(1, 3, figsize=(18, 6))

# Prepare data for plotting - get configs present in both datasets
configs_100ep_all = {c: d for c, d in all_analysis_results.items() if c[3] == 100}
configs_100ep_filtered = {c: d for c, d in filtered_results.items() if c[3] == 100}
common_configs = sorted(set(configs_100ep_all.keys()) & set(configs_100ep_filtered.keys()))

if common_configs:
    labels = []
    # All features metrics
    all_mean_max_cos = []
    all_frac_paired = []
    all_frac_shared = []
    # Filtered (alive) features metrics
    filtered_mean_max_cos = []
    filtered_frac_paired = []
    filtered_frac_shared = []
    
    for config in common_configs:
        l1, wreg_type, wreg_val, epochs, tied = config
        label = f"{wreg_type}={wreg_val}"
        if tied:
            label += " (tied)"
        labels.append(label)
        
        # All features - average across seed pairs
        all_data = configs_100ep_all[config]
        all_mean_max_cos.append(np.mean([m['mean_max_cos_avg'] for m in all_data['results'].values()]))
        all_frac_paired.append(np.mean([m['frac_paired_dec'] for m in all_data['results'].values()]))
        all_frac_shared.append(np.mean([m['frac_shared'] for m in all_data['results'].values()]))
        
        # Filtered features - average across seed pairs (filter out NaN values)
        filtered_data = configs_100ep_filtered[config]
        valid_metrics = [m for m in filtered_data['results'].values() if not np.isnan(m['mean_max_cos_avg'])]
        filtered_mean_max_cos.append(np.mean([m['mean_max_cos_avg'] for m in valid_metrics]) if valid_metrics else 0)
        filtered_frac_paired.append(np.mean([m['frac_paired_dec'] for m in valid_metrics]) if valid_metrics else 0)
        filtered_frac_shared.append(np.mean([m['frac_shared'] for m in valid_metrics]) if valid_metrics else 0)
    
    x = np.arange(len(labels))
    width = 0.35
    
    # Mean Max Cosine Similarity
    bars1a = axes[0].bar(x - width/2, all_mean_max_cos, width, color='steelblue', edgecolor='black', 
                         label='All Features', alpha=0.8)
    bars1b = axes[0].bar(x + width/2, filtered_mean_max_cos, width, color='darkorange', edgecolor='black', 
                         label=f'Alive Only (align>{ALIGNMENT_THRESHOLD})', alpha=0.8)
    axes[0].set_ylabel('Mean Max Cosine Similarity')
    axes[0].set_xlabel('Configuration')
    axes[0].set_title('Primary Metric: Mean Max Cos Sim')
    axes[0].set_xticks(x)
    axes[0].set_xticklabels(labels, rotation=45, ha='right')
    axes[0].set_ylim(0, 1)
    axes[0].legend(loc='upper left', fontsize=9)
    
    # Fraction Paired (>0.7)
    bars2a = axes[1].bar(x - width/2, all_frac_paired, width, color='steelblue', edgecolor='black', 
                         label='All Features', alpha=0.8)
    bars2b = axes[1].bar(x + width/2, filtered_frac_paired, width, color='darkorange', edgecolor='black', 
                         label=f'Alive Only (align>{ALIGNMENT_THRESHOLD})', alpha=0.8)
    axes[1].set_ylabel('Fraction Paired (>0.7)')
    axes[1].set_xlabel('Configuration')
    axes[1].set_title('Features with Max Sim > 0.7')
    axes[1].set_xticks(x)
    axes[1].set_xticklabels(labels, rotation=45, ha='right')
    axes[1].set_ylim(0, 1)
    axes[1].legend(loc='upper left', fontsize=9)
    
    # Fraction Shared (conservative)
    bars3a = axes[2].bar(x - width/2, all_frac_shared, width, color='steelblue', edgecolor='black', 
                         label='All Features', alpha=0.8)
    bars3b = axes[2].bar(x + width/2, filtered_frac_shared, width, color='darkorange', edgecolor='black', 
                         label=f'Alive Only (align>{ALIGNMENT_THRESHOLD})', alpha=0.8)
    axes[2].set_ylabel('Fraction Shared')
    axes[2].set_xlabel('Configuration')
    axes[2].set_title('Shared (Enc+Dec agree, >0.7)')
    axes[2].set_xticks(x)
    axes[2].set_xticklabels(labels, rotation=45, ha='right')
    axes[2].set_ylim(0, 1)
    axes[2].legend(loc='upper left', fontsize=9)
    
    plt.suptitle(f'Feature Consistency Comparison (100 epochs)\nBlue: All Features | Orange: Alive Only (Enc-Dec Alignment > {ALIGNMENT_THRESHOLD})', y=1.02)
    plt.tight_layout()
    plt.show()
else:
    print("No 100-epoch configurations found in both datasets")


## 15. Dead Feature Analysis Summary

**Key Findings:**

1. **Encoder-Decoder Alignment** serves as a proxy for feature "aliveness":
   - Alignment > 0.2 = feature is likely real (encoder and decoder agree on direction)
   - Alignment < 0.2 = feature may be dead or poorly trained (encoder/decoder diverged)

2. **Methodology**:
   - Uses the SAME methodology as the main analysis (Braun et al.): Mean Max Cos Sim, Hungarian matching, Shared Features
   - Conservative threshold of 0.2 to only filter truly dead features

3. **Impact on Consistency Metrics**:
   - If filtering improves consistency → dead features were adding noise
   - The "true" consistency of meaningful features may be higher than initially reported

4. **Recommendations**:
   - Report both filtered and unfiltered metrics
   - Investigate what causes dead features (initialization, sparsity penalty, etc.)
   - Consider this when comparing SAE training methods


In [ ]:
# Select one random seed and load three model variants for comparison
import random
import matplotlib.pyplot as plt

# Pick one seed that exists across all configurations
example_seed = random.choice([1,2])
print(f"Analyzing seed: {example_seed}")
tied_init = False
# Define the three configurations to compare (assuming l1=0.1, epochs=100, tied_init=True)
configs_to_compare = [
    ('No weight reg (lw=0)', (0.1, 'l1w', 0.0, 100, tied_init)),
    ('L1 on weights e-3', (0.1, 'l1w', 0.001, 100, tied_init)),
    ('L2 on weights e-5', (0.1, 'l2w', 1.0, 100, tied_init)),
    
    
    
]

# Load models for this seed
models_to_analyze = {}
for name, config in configs_to_compare:
    if config in all_grouped:
        seed_files = all_grouped[config]
        # Find the file for our example_seed
        for seed, filepath in seed_files:
            if seed == example_seed:
                print(f"Loading {name}: {os.path.basename(filepath)}")
                models_to_analyze[name] = torch.load(filepath, map_location='cpu', weights_only=False)
                break
    else:
        print(f"Warning: Configuration for {name} not found")

print(f"\nLoaded {len(models_to_analyze)} models for comparison")

In [ ]:
# Compute and plot cosine similarity of encoder-decoder vectors for all three models, each plot separate
alignments = {}
for name, model in models_to_analyze.items():
    alignments[name] = compute_encoder_decoder_alignment(model)

big_font = 18
label_font = 24
title_font = 20
legend_font = 14
tick_font = 18

for name, alignment in alignments.items():
    fig, ax = plt.subplots(1, 1, figsize=(6, 5))
    ax.hist(alignment, bins=50, edgecolor='black', alpha=0.7, color='#87cefa')  # light blue
    ax.set_xlabel('Cosine Similarity', fontsize=label_font)
    ax.set_ylabel('Frequency', fontsize=label_font)
    
    # Removed mean line
    ax.grid(True, alpha=0.3)
    ax.tick_params(axis='both', which='major', labelsize=tick_font)
    plt.tight_layout()
    plt.show()
    
    # Print statistics
    print(f"\n{name}:")
    print(f"  Mean cosine similarity: {alignment.mean():.4f}")
    print(f"  Median cosine similarity: {np.median(alignment):.4f}")
    print(f"  Features with >0.95 similarity: {np.sum(alignment > 0.95)}")

In [ ]:
# For each model/condition, plot 4 rows of 3 feature pairs (encoder next to decoder), no text or titles.

model_names = list(models_to_analyze.keys())
first_model = models_to_analyze[model_names[0]]
We_temp = first_model.encoder.weight.data.cpu().numpy().T
n_features = We_temp.shape[1]
feature_indices = [random.randint(0, n_features-1) for _ in range(4)]

for name, model in models_to_analyze.items():
    We = model.encoder.weight.data.cpu().numpy().T  # [input_dim, hidden_dim]
    Wd = model.decoder.weight.data.cpu().numpy().T  # [hidden_dim, input_dim]
    fig, axes = plt.subplots(4, 6, figsize=(18, 12))  # 4 rows, 6 columns (3 pairs per row)

    for row_idx, feature_i in enumerate(feature_indices):
        # For each row, show 3 feature pairs (encoder+decoder)
        for pair in range(3):
            f_idx = feature_i  # same randomly selected feature for this demonstration (could randomize more)
            # Encoder
            ax_enc = axes[row_idx, pair * 2]
            enc_vec = We[:, f_idx].reshape(28, 28)
            im_enc = ax_enc.imshow(enc_vec, cmap='bwr', 
                                   vmin=-np.max(np.abs(enc_vec)), 
                                   vmax=np.max(np.abs(enc_vec)))
            ax_enc.axis('off')
            # Decoder
            ax_dec = axes[row_idx, pair * 2 + 1]
            dec_vec = Wd[f_idx, :].reshape(28, 28)
            im_dec = ax_dec.imshow(dec_vec, cmap='bwr', 
                                   vmin=-np.max(np.abs(dec_vec)), 
                                   vmax=np.max(np.abs(dec_vec)))
            ax_dec.axis('off')
    plt.tight_layout()
    plt.show()

In [ ]:
# For each model/condition, plot 12 randomly chosen features with encoder-decoder cosine similarity over a per-model threshold (encoder next to decoder), no text or titles.

# Define a per-model threshold dictionary (edit thresholds as needed)
model_thresholds = {
    name: 0.7 for name in models_to_analyze
}
# Optionally, you could set manual different values:
model_thresholds = {
     'L2 on weights e-5': 0.8,
     'L1 on weights e-3': 0.7,
     'No weight reg (lw=0)': 0.5,
#     # etc...
 }

num_features_to_show = 12  # 4 rows x 3 pairs per row

for name, model in models_to_analyze.items():
    t = model_thresholds.get(name, 0.7)
    alignment = alignments[name]
    feature_candidates = np.where(alignment > t)[0]

    if len(feature_candidates) < num_features_to_show:
        print(f"Warning: Only {len(feature_candidates)} features above threshold {t:.2f} for model {name}. Showing those available.")
        feature_indices = np.random.choice(feature_candidates, len(feature_candidates), replace=False) if len(feature_candidates) > 0 else []
    else:
        feature_indices = np.random.choice(feature_candidates, num_features_to_show, replace=False)

    We = model.encoder.weight.data.cpu().numpy().T  # [input_dim, hidden_dim]
    Wd = model.decoder.weight.data.cpu().numpy().T  # [hidden_dim, input_dim]
    fig, axes = plt.subplots(4, 6, figsize=(18, 12))  # 4 rows, 6 columns (3 pairs per row)

    for i, f_idx in enumerate(feature_indices):
        row_idx = i // 3
        pair = i % 3
        # Encoder
        ax_enc = axes[row_idx, pair * 2]
        enc_vec = We[:, f_idx].reshape(28, 28)
        im_enc = ax_enc.imshow(enc_vec, cmap='bwr',
                               vmin=-np.max(np.abs(enc_vec)),
                               vmax=np.max(np.abs(enc_vec)))
        ax_enc.axis('off')
        # Decoder
        ax_dec = axes[row_idx, pair * 2 + 1]
        dec_vec = Wd[f_idx, :].reshape(28, 28)
        im_dec = ax_dec.imshow(dec_vec, cmap='bwr',
                               vmin=-np.max(np.abs(dec_vec)),
                               vmax=np.max(np.abs(dec_vec)))
        ax_dec.axis('off')
    plt.tight_layout()
    plt.show()

In [ ]:
# Select one random seed and load three model variants for comparison
import random
import matplotlib.pyplot as plt

# Pick one seed that exists across all configurations
example_seed = random.choice([0, 1, 2])
print(f"Analyzing seed: {example_seed}")
tied_init = True
# Define the three configurations to compare (assuming l1=0.1, epochs=100, tied_init=True)
configs_to_compare = [
    ('No weight reg (lw=0)', (0.1, 'l1w', 0.0, 100, tied_init)),
    ('L1 on weights e-3', (0.1, 'l1w', 0.001, 100, tied_init)),
    ('L2 on weights e-5', (0.1, 'l2w', 1.0, 100, tied_init)),
]

# Load models for this seed
models_to_analyze = {}
for name, config in configs_to_compare:
    if config in all_grouped:
        seed_files = all_grouped[config]
        # Find the file for our example_seed
        for seed, filepath in seed_files:
            if seed == example_seed:
                print(f"Loading {name}: {os.path.basename(filepath)}")
                models_to_analyze[name] = torch.load(filepath, map_location='cpu', weights_only=False)
                break
    else:
        print(f"Warning: Configuration for {name} not found")

print(f"\nLoaded {len(models_to_analyze)} models for comparison")

In [ ]:
# Select one random seed and load three model variants for comparison (tied_init=False)
import random
import matplotlib.pyplot as plt

example_seed = random.choice([1, 2])
print(f"Analyzing seed: {example_seed}")
tied_init = False

configs_to_compare = [
    ('No weight reg (lw=0)', (0.1, 'l1w', 0.0, 100, tied_init)),
    ('L1 on weights e-3', (0.1, 'l1w', 0.001, 100, tied_init)),
    ('L2 on weights e-5', (0.1, 'l2w', 1.0, 100, tied_init)),
]

# Load models for this seed
models_to_compare = {}
for name, config in configs_to_compare:
    if config in all_grouped:
        seed_files = all_grouped[config]
        for seed, filepath in seed_files:
            if seed == example_seed:
                print(f"Loading {name}: {os.path.basename(filepath)}")
                models_to_compare[name] = torch.load(filepath, map_location='cpu', weights_only=False)
                break
    else:
        print(f"Warning: Configuration for {name} not found")

print(f"\nLoaded {len(models_to_compare)} models for comparison")

In [ ]:
print(models_to_compare['No weight reg (lw=0)'])

In [ ]:
# Load MNIST and compute top cosine similarity latents for each model
from torchvision import datasets, transforms
from torch.utils.data import DataLoader
from sklearn.metrics.pairwise import cosine_similarity

mnist_transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Lambda(lambda x: x.view(-1)),
])
mnist_val = datasets.MNIST(root=str(EXTERNAL_DATA_DIR / "mnist"), train=False, download=True, transform=mnist_transform)

def ae_collate(batch):
    xs = torch.stack([x for x, _ in batch])
    return xs, xs

loader_val = DataLoader(
    mnist_val, batch_size=512, shuffle=False, num_workers=0,
    pin_memory=False, collate_fn=ae_collate,
)

k_top = 250

def compute_top_cosines_latents(model, k=100):
    We = model.encoder.weight.data.cpu().numpy() # (n_latents, in_features)
    Wd = model.decoder.weight.data.cpu().numpy() # (out_features, n_latents)
    # For cosine similarity between encoder and decoder, align shapes: 
    # compare We[i, :] with Wd[:, i] for each latent i

    n_feats = We.shape[0]
    cos_sims = []
    for i in range(n_feats):
        e_vec = We[i, :].reshape(1, -1)
        d_vec = Wd[:, i].reshape(1, -1)
        # compute cosine similarity between encoder and decoder vector for latent i
        sim = cosine_similarity(e_vec, d_vec)[0, 0]
        cos_sims.append(sim)
    cos_sims = np.array(cos_sims)
    top_features = np.argsort(np.abs(cos_sims))[-k:][::-1].copy()
    return top_features, cos_sims

top_latents_per_model = {}
cosines_per_model = {}

for model_name, model in models_to_compare.items():
    top_features, cosines = compute_top_cosines_latents(model, k=k_top)
    top_latents_per_model[model_name] = top_features
    cosines_per_model[model_name] = cosines
    print(f"\n{model_name}:")
    print(f"  Top {k_top} latent indices (by abs cosine sim): {top_features[:10]}... (showing first 10)")
    print(f"  Top cosine similarities: {cosines[top_features[:5]]} (showing first 5)")

In [ ]:
print(models_to_compare['No weight reg (lw=0)'])

# SAE_l1w_encoder_decoder(
#   (encoder): Linear(in_features=784, out_features=1568, bias=False)
#   (decoder): Linear(in_features=1568, out_features=784, bias=False)
# )

# Compute MSEs for all models across latent selection strategies
from random import sample

n_total_latents = models_to_compare[list(models_to_compare.keys())[0]].encoder.weight.shape[0]
num_images = min(len(mnist_val), 5000)

all_results = {}
strategies = ["Full SAE", f"Top {k_top} High-Var", f"{k_top} Random", f"Without Top {k_top}"]

random_latents = sample(range(n_total_latents), k_top)

def safe_encode(model, x):
    """
    Attempt to call model.encode(x). If that fails due to missing b_dec,
    try calling model.encoder(x) with relu activation.
    """
    try:
        return model.encode(x)
    except AttributeError as e:
        if hasattr(model, "encoder"):
            # See if we need to remove subtraction (likely bugged custom .encode)
            return torch.relu(model.encoder(x))
        else:
            raise e

def safe_decode(model, z):
    """
    Attempt to call model.decode(z). If that fails due to missing b_dec,
    try calling model.decoder(z) instead.
    """
    try:
        return model.decode(z)
    except AttributeError as e:
        if hasattr(model, "decoder"):
            # Fallback to plain decoder
            return model.decoder(z)
        else:
            raise e

for model_name, model in models_to_compare.items():
    print(f"\nProcessing {model_name}...")
    model.eval()
    top_features = top_latents_per_model[model_name]
    results_mse = {s: [] for s in strategies}

    for idx in range(num_images):
        if idx % 1000 == 0:
            print(f"  Processing image {idx}/{num_images}")

        x = mnist_val[idx][0].clone().detach().float().view(1, -1)

        with torch.no_grad():
            # Full SAE
            z_full = safe_encode(model, x)
            x_hat_full = safe_decode(model, z_full)
            mse_full = F.mse_loss(x_hat_full, x).item()
            results_mse["Full SAE"].append(mse_full)

            # Top-k high variance only
            z_topk = z_full.clone()
            mask_topk = torch.zeros_like(z_topk)
            mask_topk[:, top_features] = 1.0
            z_topk = z_topk * mask_topk
            x_hat_topk = safe_decode(model, z_topk)
            mse_topk = F.mse_loss(x_hat_topk, x).item()
            results_mse[f"Top {k_top} High-Var"].append(mse_topk)

            # Random latents only
            z_rand = z_full.clone()
            mask_rand = torch.zeros_like(z_rand)
            mask_rand[:, random_latents] = 1.0
            z_rand = z_rand * mask_rand
            x_hat_rand = safe_decode(model, z_rand)
            mse_rand = F.mse_loss(x_hat_rand, x).item()
            results_mse[f"{k_top} Random"].append(mse_rand)

            # Without top-k high variance
            z_wo = z_full.clone()
            z_wo[:, top_features] = 0.0
            x_hat_wo = safe_decode(model, z_wo)
            mse_wo = F.mse_loss(x_hat_wo, x).item()
            results_mse[f"Without Top {k_top}"].append(mse_wo)

    all_results[model_name] = results_mse

print("\nDone computing MSEs for all models!")


In [ ]:
print(all_results)

In [ ]:
# --- Comparison Plot: Boxplot comparing all models across strategies ---

model_names = list(all_results.keys())
n_models = len(model_names)
n_strategies = len(strategies)

if n_models == 0:
    print("No models found in all_results. Aborting plot.")
else:
    model_colors = {
        "No weight reg (lw=0)": "#4E79A7",
        "L1 on weights e-3": "#F28E2B",
        "L2 on weights e-5": "#59A14F",
    }

    fig, ax = plt.subplots(figsize=(14, 7))

    group_width = 0.8
    box_width = group_width / n_models if n_models != 0 else group_width
    positions = []

    for strat_idx in range(n_strategies):
        base_pos = strat_idx * (n_models + 1)
        for model_idx in range(n_models):
            positions.append(base_pos + model_idx * box_width)

    all_data = []
    all_colors = []

    for strat in strategies:
        for model_name in model_names:
            all_data.append(all_results[model_name][strat])
            all_colors.append(model_colors.get(model_name, "#999999"))

    if all_data:
        bp = ax.boxplot(
            all_data, 
            positions=positions, 
            widths=box_width*0.8,
            patch_artist=True, 
            notch=True,
            boxprops=dict(linewidth=2),
            medianprops=dict(linewidth=2),
            whiskerprops=dict(linewidth=1.8),
            capprops=dict(linewidth=1.8),
            flierprops=dict(marker='o', markersize=8, alpha=0.4)
        )

        for patch, color in zip(bp['boxes'], all_colors):
            patch.set_facecolor(color)
            patch.set_alpha(0.7)

        for i, (vals, pos) in enumerate(zip(all_data, positions)):
            mean_val = np.mean(vals)
            ax.scatter(pos, mean_val, marker='D', color='black', s=65, zorder=5)

        strategy_centers = []
        for strat_idx in range(n_strategies):
            base_pos = strat_idx * (n_models + 1)
            center = base_pos + (n_models - 1) * box_width / 2
            strategy_centers.append(center)

        ax.set_xticks(strategy_centers)
        ax.set_xticklabels(strategies, fontsize=22)

        legend_patches = [
            plt.Rectangle((0,0),1,1, facecolor=model_colors.get(m, "#999999"), alpha=0.7, label=m)
            for m in model_names
        ]
        ax.legend(
            handles=legend_patches, 
            loc='upper right', 
            fontsize=17, 
            frameon=True,
            markerscale=1.15
        )

        ax.set_ylabel("MSE", fontsize=22)
        ax.set_xlabel("Latent Selection Strategy", fontsize=22)
        ax.set_title(
            f"Reconstruction MSE: SAE Models x Latent Selection Strategies\n"
            f"(Top {k_top} latents by cosine similarity, seed={example_seed}, tied_init={tied_init})",
            fontsize=24,
            pad=22
        )
        ax.tick_params(axis='y', labelsize=18)
        ax.grid(axis='y', linestyle='--', alpha=0.55)

        plt.tight_layout()
        plt.show()
    else:
        print("No data to plot in boxplot.")

    # Print summary statistics
    print("\n" + "="*80)
    print("Summary Statistics (Mean +/- Std)")
    print("="*80)
    for strat in strategies:
        print(f"\n{strat}:")
        for model_name in model_names:
            vals = all_results[model_name][strat]
            print(f"  {model_name:25s}: {np.mean(vals):.6f} +/- {np.std(vals):.6f}")